## New EXP

In [ ]:
"""
Pretrained Vision Transformer (ViT)
"""

import os
import glob
import math
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from PIL import Image
from tqdm import tqdm
import timm
import wandb
import random
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# CONFIGURATION

RANDOM_STATE = 42

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
random.seed(hash("setting random seeds") % 2**32 - 1)
np.random.seed(hash("improves reproducibility") % 2**32 - 1)
torch.manual_seed(hash("by removing stochasticity") % 2**32 - 1)
torch.cuda.manual_seed_all(hash("so runs are repeatable") % 2**32 - 1)

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

# Data directory - update for your environment
DATA_DIR = "/content/drive/MyDrive/Dataset/castor_v2_224x224"
FREEZE_EARLY_LAYERS = True
PROGRESSIVE_UNFREEZING = True  # Unfreeze layers gradually during training

Using device: cuda:0


In [ ]:
# CONFIGURATION

CONFIG = {
    'batch_size': 32,
    'img_size': 224,
    'num_classes': 3,
    'num_blocks_to_keep': 8,   # Keep 8 out of 12 blocks
    'freeze_blocks': 2,
    'dropout': 0.1,
    'learning_rate': 3e-4,
    'min_lr': 1e-6,
    'warmup_epochs': 5,
    'weight_decay': 0.01,
    'epochs': 100,
    'patience': 20,
    'n_splits': 5,
    'unfreeze_epoch': 20,
    "usewandb":False
}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# PRETRAINED VISION TRANSFORMER

class PretrainedViT(nn.Module):
    def __init__(self, num_classes=3, num_blocks_to_keep=8,
                 freeze_blocks=2, dropout=0.1):
        super().__init__()

        # Load pretrained ViT-Tiny
        self.vit = timm.create_model(
            'vit_tiny_patch16_224',
            pretrained=True,
            num_classes=0,
            drop_rate=dropout,
        )

        self.embed_dim = 192
        self.freeze_blocks = freeze_blocks

        # Reduce blocks
        original_blocks = len(self.vit.blocks)
        if num_blocks_to_keep < original_blocks:
            self.vit.blocks = self.vit.blocks[:num_blocks_to_keep]
            print(f"Reduced blocks: {original_blocks} → {num_blocks_to_keep}")

        # Initial freezing
        self._freeze_layers(freeze_blocks)

        # Classification head - simpler for better convergence
        self.head = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Dropout(dropout),
            nn.Linear(self.embed_dim, num_classes)
        )

        self._init_head()

    def _freeze_layers(self, num_blocks):
        """Freeze patch embedding and first N blocks"""
        if num_blocks > 0:
            # Freeze patch embedding
            for param in self.vit.patch_embed.parameters():
                param.requires_grad = False

            # Freeze cls_token and pos_embed
            self.vit.cls_token.requires_grad = False
            self.vit.pos_embed.requires_grad = False

            # Freeze first N blocks
            for i in range(min(num_blocks, len(self.vit.blocks))):
                for param in self.vit.blocks[i].parameters():
                    param.requires_grad = False

            print(f"Frozen: patch_embed + first {num_blocks} blocks")

    def unfreeze_one_block(self):
        """Unfreeze one more block (progressive unfreezing)"""
        if self.freeze_blocks > 0:
            # Unfreeze the last frozen block
            block_to_unfreeze = self.freeze_blocks - 1
            for param in self.vit.blocks[block_to_unfreeze].parameters():
                param.requires_grad = True
            self.freeze_blocks -= 1
            print(f"Unfroze block {block_to_unfreeze}, now {self.freeze_blocks} blocks frozen")
            return True
        return False

    def _init_head(self):
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.vit.forward_features(x)
        cls_token = x[:, 0]
        logits = self.head(cls_token)
        return logits

    def get_num_params(self, trainable_only=False):
        if trainable_only:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        return sum(p.numel() for p in self.parameters())


def create_model(config):
    model = PretrainedViT(
        num_classes=config['num_classes'],
        num_blocks_to_keep=config['num_blocks_to_keep'],
        freeze_blocks=config['freeze_blocks'] if FREEZE_EARLY_LAYERS else 0,
        dropout=config['dropout'],
    )

    total_params = model.get_num_params(trainable_only=False)
    trainable_params = model.get_num_params(trainable_only=True)

    print(f"\n{'='*50}")
    print(f"Model: Pretrained ViT-Tiny (reduced)")
    print(f"Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
    print(f"Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
    print(f"{'='*50}\n")

    return model

In [ ]:
# ============================================================================
# DATASET
# ============================================================================
class ImageDataset(Dataset):
    def __init__(self, image_paths, labels, img_size=224):
        self.image_paths = image_paths
        self.labels = labels
        self.img_size = img_size
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        image = np.array(image, dtype=np.float32) / 255.0
        image = (image - self.mean) / self.std
        image = torch.from_numpy(image).permute(2, 0, 1)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label


def get_dataloaders(data_dir, batch_size, n_splits, fold_index, img_size=224):
    image_paths = []
    for ext in ['*.jpg', '*.JPG', '*.jpeg', '*.JPEG', '*.png', '*.PNG']:
        image_paths.extend(glob.glob(os.path.join(data_dir, '*', ext)))

    print(f"Found {len(image_paths)} images")

    labels = [os.path.basename(os.path.dirname(p)) for p in image_paths]
    label_encoder = LabelEncoder()
    encoded_labels = label_encoder.fit_transform(labels)
    classes = label_encoder.classes_

    print(f"Classes: {classes}")
    print(f"Number of classes: {len(classes)}")

    image_paths = np.array(image_paths)
    encoded_labels = np.array(encoded_labels)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    folds = list(skf.split(image_paths, encoded_labels))

    train_idx, val_idx = folds[fold_index]

    train_paths = image_paths[train_idx]
    train_labels = encoded_labels[train_idx]
    val_paths = image_paths[val_idx]
    val_labels = encoded_labels[val_idx]

    print(f"\nFold {fold_index + 1}/{n_splits}:")
    print(f"  Train: {len(train_paths)}, Val: {len(val_paths)}")

    for split_name, split_labels in [("Train", train_labels), ("Val", val_labels)]:
        print(f"  {split_name} distribution:")
        for cls_idx, cls_name in enumerate(classes):
            count = np.sum(split_labels == cls_idx)
            pct = count / len(split_labels) * 100
            print(f"    {cls_name}: {count} ({pct:.1f}%)")

    train_dataset = ImageDataset(train_paths, train_labels, img_size)
    val_dataset = ImageDataset(val_paths, val_labels, img_size)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=False
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True
    )

    return train_loader, val_loader, len(classes), classes

In [ ]:
# ============================================================================
# TRAINING UTILITIES
# ============================================================================
class EarlyStopping:
    def __init__(self, patience=20, min_delta=0.001, mode='max'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, score, model):
        if self.mode == 'max':
            is_better = self.best_score is None or score > self.best_score + self.min_delta
        else:
            is_better = self.best_score is None or score < self.best_score - self.min_delta

        if is_better:
            self.best_score = score
            self.counter = 0
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

    def load_best_model(self, model):
        if self.best_model_state is not None:
            model.load_state_dict(self.best_model_state)

In [ ]:
# ============================================================================
# TRAINING FUNCTION
# ============================================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Training", leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        pbar.set_postfix({'loss': loss.item(), 'acc': correct/total})

    return total_loss / total, correct / total

In [ ]:
@torch.no_grad()
def validate(model, loader, criterion, device):
    """FIXED: Properly collect predictions and labels"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        probs = F.softmax(outputs, dim=-1)
        _, predicted = outputs.max(1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    try:
        roc_auc = roc_auc_score(all_labels, all_probs, average='macro', multi_class='ovr')
    except:
        roc_auc = 0.0

    return {
        'loss': total_loss / len(all_labels),
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'roc_auc': roc_auc,
        'all_preds': all_preds,
        'all_labels': all_labels
    }


In [ ]:
def train_kfold(data_dir, config):
    results = []
    all_fold_data = []

    PROJECT_NAME = "ViT_Reduced_imagenet"
    GROUP_NAME = "Improved_NoLabelSmoothing"

    for fold in range(config['n_splits']):
        print(f"\n{'='*60}")
        print(f"FOLD {fold + 1}/{config['n_splits']}")
        print(f"{'='*60}")

        run_name = f"{GROUP_NAME}_Fold-{fold + 1}"
        if CONFIG.get("usewandb"):
            wandb.init(
                project=PROJECT_NAME,
                name=run_name,
                group=GROUP_NAME,
                config=config,
                reinit=True
            )

        train_loader, val_loader, num_classes, class_names = get_dataloaders(
            data_dir, config['batch_size'], config['n_splits'],
            fold, config['img_size']
        )

        model = create_model(config).to(device)

        if fold == 0:
            print(f"\nTotal parameters: {model.get_num_params():,}")
            print(f"Trainable parameters: {model.get_num_params(trainable_only=True):,}")

        criterion = nn.CrossEntropyLoss()

        optimizer = AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=config['learning_rate'],
            weight_decay=config['weight_decay'],
            betas=(0.9, 0.999)
        )
        scheduler = ReduceLROnPlateau(
                optimizer,mode='max',factor=0.3,patience=3)

        early_stopping = EarlyStopping(
            patience=config['patience'],
            min_delta=0.001,
            mode='max'
        )

        best_val_acc = 0
        best_epoch = 0
        best_loss = float('inf')
        best_prec = 0
        best_rec = 0
        best_f1 = 0
        best_roc = 0
        best_val_cm = None
        best_val_report = None

        # Training loop
        for epoch in range(config['epochs']):
            print(f"\nEpoch {epoch + 1}/{config['epochs']}")

            # Train
            train_loss, train_acc = train_one_epoch(
                model, train_loader, criterion, optimizer, device
            )

            # Validate
            val_metrics = validate(model, val_loader, criterion, device)

            print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
            print(f"  Val Loss: {val_metrics['loss']:.4f}, Val Acc: {val_metrics['accuracy']:.4f}")
            print(f"  Val F1: {val_metrics['f1_score']:.4f}, Val ROC-AUC: {val_metrics['roc_auc']:.4f}")

            scheduler.step(val_metrics['accuracy'])

            # To track current learning rate
            current_lr = optimizer.param_groups[0]['lr']
            print(f"  LR: {current_lr:.2e}")


            # Log to WandB
            if CONFIG.get("usewandb"):
                wandb.log({
                    'epoch': epoch + 1,
                    'learning_rate': current_lr,
                    'train_loss': train_loss,
                    'train_accuracy': train_acc,
                    'val_loss': val_metrics['loss'],
                    'val_accuracy': val_metrics['accuracy'],
                    'val_precision': val_metrics['precision'],
                    'val_recall': val_metrics['recall'],
                    'val_f1': val_metrics['f1_score'],
                    'val_auc': val_metrics['roc_auc']
                })

            # Early stopping check
            early_stopping(val_metrics['accuracy'], model)

            # Track best metrics
            if val_metrics['accuracy'] > best_val_acc:
                best_val_acc = val_metrics['accuracy']
                best_epoch = epoch + 1
                best_loss = val_metrics['loss']
                best_prec = val_metrics['precision']
                best_rec = val_metrics['recall']
                best_f1 = val_metrics['f1_score']
                best_roc = val_metrics['roc_auc']

                # Save confusion matrix and report
                if 'all_labels' in val_metrics and 'all_preds' in val_metrics:
                    best_val_cm = confusion_matrix(val_metrics['all_labels'], val_metrics['all_preds'])
                    best_val_report = classification_report(
                        val_metrics['all_labels'],
                        val_metrics['all_preds'],
                        target_names=class_names,
                        zero_division=0
                    )

                # Save model
                best_model_path = f'ViTPretrainedReducedFreeze_{GROUP_NAME}_fold{fold+1}.pth'
                torch.save(model.state_dict(), best_model_path)
                if CONFIG.get("usewandb"):
                    wandb.save(best_model_path)
                print(f"  ✓ Best model saved (acc: {best_val_acc:.4f})")

            if early_stopping.early_stop:
                print(f"\n  Early stopping at epoch {epoch + 1}")
                break

        # Load best model
        early_stopping.load_best_model(model)

        # Get final validation metrics
        final_val_metrics = validate(model, val_loader, criterion, device)

        # Log confusion matrix if available
        if best_val_cm is not None:
            plt.figure(figsize=(8, 6))
            sns.heatmap(best_val_cm, annot=True, fmt="d", cmap="Blues",
                       xticklabels=class_names, yticklabels=class_names)
            plt.xlabel("Predicted")
            plt.ylabel("True")
            plt.title(f"Fold {fold+1} - Validation Confusion Matrix")
            val_cm_img = f"Fold{fold+1}_Validation_CM.png"
            plt.savefig(val_cm_img, dpi=300, bbox_inches="tight")
            plt.close()
            if CONFIG.get("usewandb"):
                wandb.log({
                    "best_validation_confusion_matrix": wandb.Image(val_cm_img),
                    "best_validation_classification_report": wandb.Html(f"<pre>{best_val_report}</pre>")
                })

        # Store fold results - THIS WAS MISSING!
        fold_result = {
            'fold': fold + 1,
            'best_epoch': best_epoch,
            'accuracy': best_val_acc,
            'loss': best_loss,
            'precision': best_prec,
            'recall': best_rec,
            'f1_score': best_f1,
            'roc_auc': best_roc
        }
        results.append(fold_result)

        # Store for table
        all_fold_data.append([
            fold + 1, best_epoch, best_val_acc, best_loss,
            best_prec, best_rec, best_f1, best_roc
        ])
        if CONFIG.get("usewandb"):
            # Update wandb summary for this fold
            wandb.summary["fold"] = fold + 1
            wandb.summary["best_epoch"] = best_epoch
            wandb.summary["best_val_loss"] = best_loss
            wandb.summary["final_val_accuracy"] = best_val_acc
            wandb.summary["final_precision"] = best_prec
            wandb.summary["final_recall"] = best_rec
            wandb.summary["final_f1"] = best_f1
            wandb.summary["final_roc_auc"] = best_roc

        # Cleanup
        del model, train_loader, val_loader, optimizer, scheduler
        torch.cuda.empty_cache()
        gc.collect()
        if CONFIG.get("usewandb"):
            wandb.finish()

    # FIXED: Log summary table after all folds complete
    if CONFIG.get("usewandb"):
        wandb.init(
            project=PROJECT_NAME,
            name=f"{GROUP_NAME}_Summary",
            group=GROUP_NAME,
            job_type="summary",
            config=config
        )

        summary_table = wandb.Table(
            columns=["Fold", "Best Epoch", "Accuracy", "Loss",
                    "Precision", "Recall", "F1", "ROC-AUC"],
            data=all_fold_data
        )
        wandb.log({"all_folds_summary": summary_table})

    # Summary statistics
    print(f"\n{'='*60}")
    print("K-FOLD CROSS VALIDATION SUMMARY")
    print(f"{'='*60}")

    for r in results:
        print(f"Fold {r['fold']}: Acc={r['accuracy']:.4f}, F1={r['f1_score']:.4f}, "
              f"ROC-AUC={r['roc_auc']:.4f}")

    # Calculate mean and std for each metric
    print(f"\nMean Results:")
    for metric in ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']:
        values = [r[metric] for r in results]
        mean_val = np.mean(values)
        std_val = np.std(values)
        print(f"  {metric}: {mean_val:.4f} ± {std_val:.4f}")

        # Log to wandb summary
        if CONFIG.get("usewandb"):
            wandb.summary[f"mean_{metric}"] = mean_val
            wandb.summary[f"std_{metric}"] = std_val
    if CONFIG.get("usewandb"):
        wandb.finish()

    return results


In [ ]:
# ============================================================================
# MAIN
# ============================================================================
if __name__ == "__main__":
    print("="*60)
    print("VISION TRANSFORMER TRAINING - PYTORCH")
    print("="*60)
    print(f"Configuration:")
    for k, v in CONFIG.items():
        print(f"  {k}: {v}")
    print("="*60)

    results = train_kfold(DATA_DIR, CONFIG)


VISION TRANSFORMER TRAINING - PYTORCH
Configuration:
  batch_size: 32
  img_size: 224
  num_classes: 3
  num_blocks_to_keep: 8
  freeze_blocks: 2
  dropout: 0.1
  learning_rate: 0.0003
  min_lr: 1e-06
  warmup_epochs: 5
  weight_decay: 0.01
  epochs: 5
  patience: 20
  n_splits: 5
  unfreeze_epoch: 20
  usewandb: False

FOLD 1/5
Found 991 images
Classes: ['Fresh Leafs' 'Semilooper' 'Spodoptera']
Number of classes: 3

Fold 1/5:
  Train: 792, Val: 199
  Train distribution:
    Fresh Leafs: 382 (48.2%)
    Semilooper: 256 (32.3%)
    Spodoptera: 154 (19.4%)
  Val distribution:
    Fresh Leafs: 96 (48.2%)
    Semilooper: 64 (32.2%)
    Spodoptera: 39 (19.6%)
Reduced blocks: 12 → 8
Frozen: patch_embed + first 2 blocks

Model: Pretrained ViT-Tiny (reduced)
Total parameters: 3,745,923 (3.75M)
Trainable parameters: 2,670,531 (2.67M)


Total parameters: 3,745,923
Trainable parameters: 2,670,531

Epoch 1/5


  Train Loss: 0.3605, Train Acc: 0.8611
  Val Loss: 0.1921, Val Acc: 0.9347
  Val F1: 0.9081, Val ROC-AUC: 0.9868
  LR: 3.00e-04
  ✓ Best model saved (acc: 0.9347)

Epoch 2/5


  Train Loss: 0.1104, Train Acc: 0.9646
  Val Loss: 0.0741, Val Acc: 0.9849
  Val F1: 0.9801, Val ROC-AUC: 0.9984
  LR: 3.00e-04
  ✓ Best model saved (acc: 0.9849)

Epoch 3/5


  Train Loss: 0.0428, Train Acc: 0.9848
  Val Loss: 0.1350, Val Acc: 0.9648
  Val F1: 0.9525, Val ROC-AUC: 0.9922
  LR: 3.00e-04

Epoch 4/5


  Train Loss: 0.0247, Train Acc: 0.9924
  Val Loss: 0.0764, Val Acc: 0.9749
  Val F1: 0.9658, Val ROC-AUC: 0.9982
  LR: 3.00e-04

Epoch 5/5


...

  Train Loss: 0.3381, Train Acc: 0.8689
  Val Loss: 0.2001, Val Acc: 0.9242
  Val F1: 0.8831, Val ROC-AUC: 0.9945
  LR: 3.00e-04
  ✓ Best model saved (acc: 0.9242)

Epoch 2/5


  Train Loss: 0.1357, Train Acc: 0.9483
  Val Loss: 0.2480, Val Acc: 0.8990
  Val F1: 0.8359, Val ROC-AUC: 0.9886
  LR: 3.00e-04

Epoch 3/5


  Train Loss: 0.0590, Train Acc: 0.9823
  Val Loss: 0.0564, Val Acc: 0.9798
  Val F1: 0.9720, Val ROC-AUC: 0.9989
  LR: 3.00e-04
  ✓ Best model saved (acc: 0.9798)

Epoch 4/5


  Train Loss: 0.0345, Train Acc: 0.9887
  Val Loss: 0.1355, Val Acc: 0.9293
  Val F1: 0.9085, Val ROC-AUC: 0.9966
  LR: 3.00e-04

Epoch 5/5


  Train Loss: 0.0267, Train Acc: 0.9924
  Val Loss: 0.0435, Val Acc: 0.9848
  Val F1: 0.9789, Val ROC-AUC: 0.9993
  LR: 3.00e-04
  ✓ Best model saved (acc: 0.9848)

K-FOLD CROSS VALIDATION SUMMARY
Fold 1: Acc=0.9849, F1=0.9801, ROC-AUC=0.9984
Fold 2: Acc=0.9899, F1=0.9864, ROC-AUC=0.9998
Fold 3: Acc=0.9697, F1=0.9591, ROC-AUC=0.9972
Fold 4: Acc=0.9747, F1=0.9649, ROC-AUC=0.9976
Fold 5: Acc=0.9848, F1=0.9789, ROC-AUC=0.9993

Mean Results:
  accuracy: 0.9808 ± 0.0074
  precision: 0.9741 ± 0.0106
  recall: 0.9740 ± 0.0102
  f1_score: 0.9739 ± 0.0102
  roc_auc: 0.9985 ± 0.0010


In [ ]:
# ============================================================================
# MAIN
# ============================================================================
if __name__ == "__main__":
    print("="*60)
    print("VISION TRANSFORMER TRAINING - PYTORCH")
    print("="*60)
    print(f"Configuration:")
    for k, v in CONFIG.items():
        print(f"  {k}: {v}")
    print("="*60)

    results = train_kfold(DATA_DIR, CONFIG)


Using device: cuda
VISION TRANSFORMER TRAINING - PYTORCH
Configuration:
  batch_size: 32
  img_size: 224
  num_classes: 3
  num_blocks_to_keep: 8
  freeze_blocks: 2
  dropout: 0.1
  learning_rate: 0.0003
  min_lr: 1e-06
  warmup_epochs: 5
  weight_decay: 0.01
  epochs: 100
  patience: 20
  n_splits: 5
  unfreeze_epoch: 20

FOLD 1/5


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


...

  Train Loss: 0.0041, Train Acc: 0.9962
  Val Loss: 0.0318, Val Acc: 0.9899
  Val F1: 0.9859, Val ROC-AUC: 1.0000
  LR: 2.55e-04

Epoch 30/100


  Train Loss: 0.0043, Train Acc: 0.9975
  Val Loss: 0.0227, Val Acc: 0.9949
  Val F1: 0.9930, Val ROC-AUC: 1.0000
  LR: 2.52e-04

  Early stopping at epoch 30


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
learning_rate,▁▃▆██████████████▇▇▇▇▇▇▇▇▇▇▆▆▆
train_accuracy,▁▆▇▇██████████████████████████
train_loss,█▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▆▆▇▇▅▆▇██████████████████████
val_auc,▂▆▇▁▅▅▆▇██████████████████████
val_f1,▁▇▆▇▇▆▆▇██████████████████████
val_loss,█▄▃▅▃▅▄▄▂▁▁▁▁▁▁▂▁▁▁▁▂▁▁▁▂▂▂▁▂▁
val_precision,▁▆▆▇▇▅▆▆▇█████████████████████
val_recall,▁▇▆▇▇▆▆▇██████████████████████
best_epoch,10



K-FOLD CROSS VALIDATION SUMMARY
Fold 1: Acc=0.9799, F1=0.9728, ROC-AUC=0.9962
Fold 2: Acc=0.9899, F1=0.9861, ROC-AUC=0.9999
Fold 3: Acc=0.9798, F1=0.9774, ROC-AUC=0.9926
Fold 4: Acc=0.9848, F1=0.9789, ROC-AUC=0.9989
Fold 5: Acc=0.9949, F1=0.9930, ROC-AUC=1.0000

Mean Results:
  accuracy: 0.9859 ± 0.0059
  precision: 0.9825 ± 0.0089
  recall: 0.9813 ± 0.0055
  f1_score: 0.9816 ± 0.0071
  roc_auc: 0.9975 ± 0.0028


mean_accuracy,0.98588
mean_f1_score,0.98164
mean_precision,0.98252
mean_recall,0.98128
mean_roc_auc,0.99754
std_accuracy,0.00587
std_f1_score,0.0071
std_precision,0.00887
std_recall,0.00551
std_roc_auc,0.0028
